<a href="https://colab.research.google.com/github/marcouras/AI-engineering-fundamentals/blob/main/lezione5/Lezione5_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# 🤖 AI Engineering Fundamentals
## Lezione 5 — Tools, Function Calling & MCP

**ITS Novitas 4.0 — Sviluppatore Intelligenza Artificiale**  
Docente: Marco Uras | 📅 Giovedì 04/06/2026

---

### 🎯 Obiettivi
- ✅ Definire tool in JSON e integrarli nell'API
- ✅ Implementare il tool loop
- ✅ Costruire 3 tool reali (calcolatrice, meteo, Wikipedia)
- ✅ Capire MCP e connettersi a un server

In [2]:
!pip install anthropic requests -q
from google.colab import userdata
import anthropic, os, json, requests

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
print("✅ Setup completato!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.7/838.7 kB 18.1 MB/s eta 0:00:00
✅ Setup completato!


---
## 1. Tool Calcolatrice

Il tool più semplice — definizione JSON + funzione Python.

In [3]:
# DEFINIZIONE del tool (cosa il modello può leggere)
tool_calcolatrice = {
    "name": "calcola",
    "description": "Esegui un'operazione matematica. Usa questo tool per calcoli aritmetici precisi.",
    "input_schema": {
        "type": "object",
        "properties": {
            "espressione": {
                "type": "string",
                "description": "L'espressione matematica da calcolare. Es: '234 * 567' o '(100 + 200) / 3'"
            }
        },
        "required": ["espressione"]
    }
}

# IMPLEMENTAZIONE del tool (la funzione Python reale)
def calcola(espressione: str) -> str:
    """Valuta un'espressione matematica in modo sicuro."""
    try:
        # eval() limitato solo a operazioni matematiche
        allowed = set('0123456789+-*/().% ')
        if not all(c in allowed for c in espressione):
            return "Errore: espressione non valida"
        risultato = eval(espressione)
        return f"{espressione} = {risultato}"
    except Exception as e:
        return f"Errore nel calcolo: {str(e)}"

# Test diretto
print(calcola("234 * 567"))
print(calcola("(100 + 200) / 3"))

234 * 567 = 132678
(100 + 200) / 3 = 100.0


In [4]:
# IL TOOL LOOP — il cuore del function calling
def esegui_tool(nome, parametri):
    """Router: smista la chiamata al tool giusto."""
    if nome == "calcola":
        return calcola(parametri["espressione"])
    return f"Tool '{nome}' non trovato"

def chat_con_tool(messaggio, tools, history=None):
    """Chatbot con tool use. Gestisce il loop automaticamente."""
    if history is None:
        history = []

    history.append({"role": "user", "content": messaggio})

    while True:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=tools,
            messages=history
        )

        # Se il modello ha finito, restituisce la risposta
        if response.stop_reason == "end_turn":
            testo = next(b.text for b in response.content if b.type == "text")
            history.append({"role": "assistant", "content": response.content})
            return testo, history

        # Se il modello vuole usare un tool
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})

            # Esegui tutti i tool richiesti
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 Chiama tool: {block.name}({block.input})")
                    risultato = esegui_tool(block.name, block.input)
                    print(f"  ✅ Risultato: {risultato}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(risultato)
                    })

            # Rimanda i risultati al modello
            history.append({"role": "user", "content": tool_results})

# Test
print("❓ Domanda: Quanto fa 1234 * 5678?")
risposta, history = chat_con_tool("Quanto fa 1234 * 5678?", [tool_calcolatrice])
print(f"\n🤖 Risposta: {risposta}")

❓ Domanda: Quanto fa 1234 * 5678?
  🔧 Chiama tool: calcola({'espressione': '1234 * 5678'})
  ✅ Risultato: 1234 * 5678 = 7006652

🤖 Risposta: Il risultato è **7.006.652**

1234 × 5678 = 7.006.652


---
## 2. Tool Meteo (API reale)

Usiamo [open-meteo.com](https://open-meteo.com) — gratuito, senza API key.

In [5]:
# Coordinate delle città sarde principali
CITTA_COORD = {
    "sassari": {"lat": 40.7259, "lon": 8.5563},
    "cagliari": {"lat": 39.2238, "lon": 9.1217},
    "nuoro": {"lat": 40.3207, "lon": 9.3311},
    "oristano": {"lat": 39.9069, "lon": 8.5889},
    "olbia": {"lat": 40.9237, "lon": 9.4992},
    "roma": {"lat": 41.9028, "lon": 12.4964},
    "milano": {"lat": 45.4642, "lon": 9.1900},
}

tool_meteo = {
    "name": "get_meteo",
    "description": "Ottieni il meteo attuale e le previsioni per una città italiana.",
    "input_schema": {
        "type": "object",
        "properties": {
            "citta": {
                "type": "string",
                "description": "Nome della città in minuscolo (es: 'sassari', 'cagliari', 'roma')"
            }
        },
        "required": ["citta"]
    }
}

def get_meteo(citta: str) -> str:
    citta = citta.lower().strip()
    if citta not in CITTA_COORD:
        return f"Città '{citta}' non trovata. Disponibili: {', '.join(CITTA_COORD.keys())}"

    coord = CITTA_COORD[citta]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={coord['lat']}&longitude={coord['lon']}"
        f"&current=temperature_2m,weathercode,windspeed_10m,relative_humidity_2m"
        f"&daily=temperature_2m_max,temperature_2m_min,precipitation_sum"
        f"&timezone=Europe/Rome&forecast_days=3"
    )
    try:
        data = requests.get(url, timeout=5).json()
        curr = data["current"]
        daily = data["daily"]
        return (
            f"Meteo a {citta.title()} ora: {curr['temperature_2m']}°C, "
            f"umidità {curr['relative_humidity_2m']}%, vento {curr['windspeed_10m']} km/h. "
            f"Prossimi 3 giorni: "
            f"oggi max {daily['temperature_2m_max'][0]}°C min {daily['temperature_2m_min'][0]}°C, "
            f"domani max {daily['temperature_2m_max'][1]}°C, "
            f"dopodomani max {daily['temperature_2m_max'][2]}°C."
        )
    except Exception as e:
        return f"Errore API meteo: {str(e)}"

print(get_meteo("Londra"))
print(get_meteo("Sassari"))

Città 'londra' non trovata. Disponibili: sassari, cagliari, nuoro, oristano, olbia, roma, milano
Meteo a Sassari ora: 27.8°C, umidità 41%, vento 11.4 km/h. Prossimi 3 giorni: oggi max 28.4°C min 15.6°C, domani max 27.9°C, dopodomani max 25.9°C.


---
## 3. Tool Wikipedia


In [6]:
tool_wikipedia = {
    "name": "cerca_wikipedia",
    "description": "Cerca informazioni su Wikipedia. Usa per fatti, biografie, concetti tecnici, eventi storici.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Termine da cercare su Wikipedia (in italiano o inglese)"
            },
            "lingua": {
                "type": "string",
                "description": "Lingua Wikipedia: 'it' per italiano, 'en' per inglese",
                "enum": ["it", "en"]
            }
        },
        "required": ["query"]
    }
}

HEADERS = {
    "User-Agent": "WiDataChatbot/1.0 (corso ITS Novitas; marco@widata.cloud)"
}

def cerca_wikipedia(query: str, lingua: str = "it") -> str:
    try:
        url = f"https://{lingua}.wikipedia.org/api/rest_v1/page/summary/{query.replace(' ', '_')}"
        r = requests.get(url, timeout=5, headers=HEADERS)

        if r.status_code == 404:
            search = requests.get(
                f"https://{lingua}.wikipedia.org/w/api.php",
                params={"action": "opensearch", "search": query,
                        "format": "json", "limit": 1},
                timeout=5,
                headers=HEADERS
            )
            if search.status_code != 200 or not search.text.strip():
                return f"Nessun risultato per '{query}'"

            dati_search = search.json()
            if not dati_search[1]:
                return f"Nessun risultato per '{query}' su Wikipedia {lingua}"

            titolo = dati_search[1][0]
            r = requests.get(
                f"https://{lingua}.wikipedia.org/api/rest_v1/page/summary/{titolo.replace(' ', '_')}",
                timeout=5,
                headers=HEADERS
            )

        if r.status_code != 200:
            return f"Wikipedia non disponibile (status {r.status_code})"
        if not r.text.strip():
            return f"Wikipedia ha risposto vuoto per '{query}'"

        data = r.json()
        extract = data.get("extract", "Nessuna descrizione disponibile")
        if len(extract) > 500:
            extract = extract[:500] + "..."
        return f"Wikipedia ({data.get('title', query)}): {extract}"

    except requests.Timeout:
        return "Errore: Wikipedia non risponde (timeout)"
    except Exception as e:
        return f"Errore Wikipedia: {str(e)}"

print(cerca_wikipedia("Sassari", lingua="it"))

Wikipedia (Sassari): Sassari è un comune italiano di 120 185 abitanti, capoluogo dell'omonima città metropolitana in Sardegna.


---
## 4. Chatbot con tutti i tool

In [7]:
# Router aggiornato con tutti i tool
TUTTI_I_TOOL = [tool_calcolatrice, tool_meteo, tool_wikipedia]

def esegui_tool_v2(nome, parametri):
    if nome == "calcola":       return calcola(parametri["espressione"])
    if nome == "get_meteo":     return get_meteo(parametri["citta"])
    if nome == "cerca_wikipedia": return cerca_wikipedia(parametri["query"], parametri.get("lingua", "it"))
    return f"Tool '{nome}' non trovato"

def chat_multi_tool(messaggio):
    history = [{"role": "user", "content": messaggio}]

    while True:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=TUTTI_I_TOOL,
            messages=history
        )

        if response.stop_reason == "end_turn":
            return next(b.text for b in response.content if b.type == "text")

        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({block.input})")
                    risultato = esegui_tool_v2(block.name, block.input)
                    tool_results.append({"type":"tool_result","tool_use_id":block.id,"content":str(risultato)})
            history.append({"role": "user", "content": tool_results})

# Test con domande che richiedono tool diversi
domande = [
    "Quanto fa 15% di 847?",
    "Che tempo fa a Sassari oggi?",
    "Chi ha fondato Anthropic?",
    "Fa più caldo a Sassari o a Cagliari oggi?",  # usa il meteo 2 volte!
]

for d in domande:
    print(f"\n{'='*50}")
    print(f"❓ {d}")
    r = chat_multi_tool(d)
    print(f"🤖 {r}")


❓ Quanto fa 15% di 847?
  🔧 calcola({'espressione': '847 * 0.15'})
🤖 Il **15% di 847 è 127,05**.

❓ Che tempo fa a Sassari oggi?
  🔧 get_meteo({'citta': 'sassari'})
🤖 Il tempo a Sassari oggi è:

- **Temperatura attuale**: 27.8°C
- **Umidità**: 41%
- **Vento**: 11.4 km/h
- **Temperature previste per oggi**: 
  - Massima: 28.4°C
  - Minima: 15.6°C

**Previsioni per i prossimi giorni**:
- Domani: massima 27.9°C
- Dopodomani: massima 25.9°C

Insomma, una giornata abbastanza piacevole con temperature miti! 😊

❓ Chi ha fondato Anthropic?
  🔧 cerca_wikipedia({'query': 'Anthropic', 'lingua': 'it'})
🤖 Anthropic è stata fondata nel **2021** da un gruppo di ricercatori e imprenditori nel campo dell'intelligenza artificiale, tra cui:

- **Dario Amodei** (CEO)
- **Daniela Amodei** (Presidente)
- E altri ricercatori provenienti da OpenAI e altre istituzioni

La società è una benefit corporation con sede a San Francisco ed è focalizzata sullo sviluppo di sistemi di intelligenza artificiale sicuri e 

---
## 5. MCP — Model Context Protocol

MCP standardizza come i tool si espongono ai modelli AI. Ogni server MCP
segue lo stesso protocollo — quindi funziona con Claude, GPT, Gemini e altri.

Per questo notebook usiamo una **simulazione** del pattern MCP, che mostra
come un server MCP espone i tool. La connessione reale a server MCP si
configura nel client Claude desktop o in applicazioni dedicate.

In [8]:
# Simulazione del pattern MCP — come un server MCP espone i tool

class MockMCPServer:
    """Simula un server MCP per il filesystem locale."""

    def list_tools(self):
        """Restituisce la lista dei tool disponibili (come farebbe un server MCP reale)."""
        return [
            {
                "name": "read_file",
                "description": "Leggi il contenuto di un file di testo",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "Percorso del file"}
                    },
                    "required": ["path"]
                }
            },
            {
                "name": "list_files",
                "description": "Elenca i file in una directory",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "directory": {"type": "string", "description": "Percorso della directory"}
                    },
                    "required": ["directory"]
                }
            }
        ]

    def call_tool(self, nome, parametri):
        """Esegue un tool e restituisce il risultato."""
        import os
        if nome == "read_file":
            path = parametri["path"]
            if os.path.exists(path):
                with open(path, "r", encoding="utf-8") as f:
                    content = f.read()
                return content[:1000]  # max 1000 caratteri
            return f"File non trovato: {path}"
        elif nome == "list_files":
            directory = parametri["directory"]
            if os.path.exists(directory):
                files = os.listdir(directory)
                return f"File in '{directory}': {', '.join(files[:20])}"
            return f"Directory non trovata: {directory}"
        return f"Tool '{nome}' non disponibile"

# Inizializza il server MCP
mcp_server = MockMCPServer()

# Mostra i tool disponibili (come farebbe un client MCP)
print("🔌 Tool disponibili dal server MCP:")
for tool in mcp_server.list_tools():
    print(f"  • {tool['name']}: {tool['description']}")

🔌 Tool disponibili dal server MCP:
  • read_file: Leggi il contenuto di un file di testo
  • list_files: Elenca i file in una directory


In [9]:
# Integra i tool MCP nel chatbot
mcp_tools = mcp_server.list_tools()
tutti_i_tool_con_mcp = TUTTI_I_TOOL + mcp_tools

def esegui_tool_completo(nome, parametri):
    """Router che gestisce sia tool custom che tool MCP."""
    # Tool custom
    if nome == "calcola":           return calcola(parametri["espressione"])
    if nome == "get_meteo":         return get_meteo(parametri["citta"])
    if nome == "cerca_wikipedia":   return cerca_wikipedia(parametri["query"], parametri.get("lingua", "it"))
    # Tool MCP
    if nome in ["read_file", "list_files"]:
        return mcp_server.call_tool(nome, parametri)
    return f"Tool '{nome}' non trovato"

# Crea un file di test
with open("note_widata.txt", "w") as f:
    f.write("WiData Srl - Note interne\n\nCliente: Comune di Sassari\nProgetto: Monitoraggio qualità aria\nSensori installati: 12 XS200\nData installazione: 15/03/2026\nStato: operativo")

def chat_con_mcp(messaggio):
    history = [{"role": "user", "content": messaggio}]
    while True:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=tutti_i_tool_con_mcp,
            messages=history
        )
        if response.stop_reason == "end_turn":
            return next(b.text for b in response.content if b.type == "text")
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({block.input})")
                    risultato = esegui_tool_completo(block.name, block.input)
                    tool_results.append({"type":"tool_result","tool_use_id":block.id,"content":str(risultato)})
            history.append({"role": "user", "content": tool_results})

# Test
print("❓ Leggi le note WiData e dimmi quanti sensori sono installati")
r = chat_con_mcp("Leggi il file note_widata.txt e dimmi quanti sensori XS200 sono installati")
print(f"\n🤖 {r}")

❓ Leggi le note WiData e dimmi quanti sensori sono installati
  🔧 read_file({'path': 'note_widata.txt'})

🤖 Secondo il file **note_widata.txt**, sono installati **12 sensori XS200** per il progetto di monitoraggio della qualità dell'aria del Comune di Sassari. L'installazione è stata completata il 15/03/2026 e lo stato è operativo.


---
## ⭐ Esercizi

In [10]:
NOME_STUDENTE = "Stefano"  # ← SCRIVI IL TUO NOME
if NOME_STUDENTE:
    print(f"✅ Notebook di: {NOME_STUDENTE}")
else:
    print("⚠️ Scrivi il tuo nome!")

✅ Notebook di: Stefano


### Esercizio 1 — Tool calcolatrice avanzata ★☆☆
Estendi la calcolatrice con un secondo tool `converti_unita` che converte tra unità di misura comuni (km/miglia, celsius/fahrenheit, kg/libbre). Testalo con almeno 3 conversioni.

In [11]:
# ESERCIZIO 1


def esegui_tool(nome, parametri):
    """Router: smista la chiamata al tool giusto."""
    if nome == "calcola":
        return calcola(parametri["espressione"])
    if nome == "converti_unita":
        return converti_unita(parametri["valore"], parametri["da"], parametri["a"])

    return f"Tool '{nome}' non trovato"



def chat_con_tool(messaggio, tools, history=None):
    """Chatbot con tool use. Gestisce il loop automaticamente."""
    if history is None:
        history = []

    history.append({"role": "user", "content": messaggio})

    while True:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=tools,
            messages=history
        )

        # Se il modello ha finito, restituisce la risposta
        if response.stop_reason == "end_turn":
            testo = next(b.text for b in response.content if b.type == "text")
            history.append({"role": "assistant", "content": response.content})
            return testo, history

        # Se il modello vuole usare un tool
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})

            # Esegui tutti i tool richiesti
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 Chiama tool: {block.name}({block.input})")
                    risultato = esegui_tool(block.name, block.input)
                    print(f"  ✅ Risultato: {risultato}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(risultato)
                    })

            # Rimanda i risultati al modello
            history.append({"role": "user", "content": tool_results})






tool_converti = {
    "name": "converti_unita",
    "description": "Usa questo tool quando l'utente chiede di convertire una misura in un'altra (es. chilometri in miglia, gradi Celsius in Fahrenheit o chilogrammi in libbre).",  # ← descrivi quando usarlo
    "input_schema": {
        "type": "object",
        "properties": {
            "valore": {"type": "number", "description": "Il valore da convertire"},
            "da": {"type": "string", "description": "Unità di partenza (es: 'km', 'celsius', 'kg')"},
            "a": {"type": "string", "description": "Unità di destinazione (es: 'miglia', 'fahrenheit', 'libbre')"}
        },
        "required": ["valore", "da", "a"]
    }
}





def converti_unita(valore, da, a):
    da = da.lower()
    a = a.lower()

    # l'equivalente dello switch in java
    match (da, a):
        case ("km", "miglia"):
            risultato = valore * 0.621371
        case ("miglia", "km"):
            risultato = valore * 1.60934

        case ("celsius", "fahrenheit"):
            risultato = (valore * 9/5) + 32
        case ("fahrenheit", "celsius"):
            risultato = (valore - 32) * 5/9

        case ("kg", "libbre"):
            risultato = valore * 2.20462
        case ("libbre", "kg"):
            risultato = valore * 0.453592

        case _:
            return f"Errore: la conversione da '{da}' a '{a}' non è supportata."


    return f"{valore} {da} corrispondono a {risultato:.2f} {a}"

# Test
print(chat_con_tool("Converti 100 km in miglia", [tool_calcolatrice, tool_converti]))
print(chat_con_tool("Converti 50 libbre in kg", [tool_calcolatrice, tool_converti]))
print(chat_con_tool("Converti 150 gradi celsius in farenait", [tool_calcolatrice, tool_converti]))

  🔧 Chiama tool: converti_unita({'valore': 100, 'da': 'km', 'a': 'miglia'})
  ✅ Risultato: 100 km corrispondono a 62.14 miglia
('**100 km corrispondono a 62,14 miglia**', [{'role': 'user', 'content': 'Converti 100 km in miglia'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01Eg679sWjgA3LWCT1NvujVW', caller=DirectCaller(type='direct'), input={'valore': 100, 'da': 'km', 'a': 'miglia'}, name='converti_unita', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01Eg679sWjgA3LWCT1NvujVW', 'content': '100 km corrispondono a 62.14 miglia'}]}, {'role': 'assistant', 'content': [TextBlock(citations=None, text='**100 km corrispondono a 62,14 miglia**', type='text')]}])
  🔧 Chiama tool: converti_unita({'valore': 50, 'da': 'libbre', 'a': 'kg'})
  ✅ Risultato: 50 libbre corrispondono a 22.68 kg
('50 libbre corrispondono a **22,68 kg** (chilogrammi).', [{'role': 'user', 'content': 'Converti 50 libbre in kg'}, {'role': 'assistant', 'content': [To

In [12]:
#SOLUZIONE ESERCIZIO 1

tool_converti = {
    "name": "converti_unita",
    "description": "Converti un valore tra unità di misura. Usa per conversioni km/miglia, celsius/fahrenheit, kg/libbre, metri/piedi.",
    "input_schema": {
        "type": "object",
        "properties": {
            "valore": {"type": "number", "description": "Il valore da convertire"},
            "da": {"type": "string", "description": "Unità di partenza (es: 'km', 'celsius', 'kg')"},
            "a": {"type": "string", "description": "Unità di destinazione (es: 'miglia', 'fahrenheit', 'libbre')"}
        },
        "required": ["valore", "da", "a"]
    }
}

def converti_unita(valore, da, a):
    da = da.lower().strip()
    a  = a.lower().strip()

    conversioni = {
        ("km",       "miglia"):      lambda v: v * 0.621371,
        ("miglia",   "km"):          lambda v: v * 1.60934,
        ("celsius",  "fahrenheit"):  lambda v: v * 9/5 + 32,
        ("fahrenheit","celsius"):    lambda v: (v - 32) * 5/9,
        ("kg",       "libbre"):      lambda v: v * 2.20462,
        ("libbre",   "kg"):          lambda v: v * 0.453592,
        ("metri",    "piedi"):       lambda v: v * 3.28084,
        ("piedi",    "metri"):       lambda v: v / 3.28084,
        ("km",       "metri"):       lambda v: v * 1000,
        ("metri",    "km"):          lambda v: v / 1000,
    }

    if (da, a) in conversioni:
        risultato = conversioni[(da, a)](valore)
        return f"{valore} {da} = {risultato:.4f} {a}"

    return f"Conversione '{da}' → '{a}' non supportata."

# Test diretto
print(converti_unita(100, "km", "miglia"))
print(converti_unita(37, "celsius", "fahrenheit"))
print(converti_unita(80, "kg", "libbre"))

def chat_con_tool_router(messaggio, tools, esegui_fn=None, history=None):
    if history is None:
        history = []

    history.append({"role": "user", "content": messaggio})

    while True:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=tools,
            messages=history
        )

        if response.stop_reason == "end_turn":
            return next(b.text for b in response.content if b.type == "text")

        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({block.input})")
                    risultato = esegui_fn(block.name, block.input) if esegui_fn else "Tool non implementato"
                    print(f"  ✅ {risultato}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(risultato)
                    })
            history.append({"role": "user", "content": tool_results})
# Test con il chatbot
def esegui(nome, params):
    if nome == "calcola":       return calcola(params["espressione"])
    if nome == "converti_unita": return converti_unita(params["valore"], params["da"], params["a"])
    return "Tool non trovato"

print()
print(chat_con_tool_router("Converti 100 km in miglia", [tool_calcolatrice, tool_converti], esegui))

100 km = 62.1371 miglia
37 celsius = 98.6000 fahrenheit
80 kg = 176.3696 libbre

  🔧 converti_unita({'valore': 100, 'da': 'km', 'a': 'miglia'})
  ✅ 100 km = 62.1371 miglia
100 km equivalgono a **62.1371 miglia**.


### Esercizio 2 — Multi-step con tool concatenati ★★☆
Fai una domanda che richiede almeno 2 tool in sequenza. Es: 'Qual è la temperatura in Fahrenheit a Sassari?' (meteo → conversione). Verifica che il modello concateni i tool correttamente.

In [13]:
# ESERCIZIO 2

TUTTI_I_TOOL = [tool_calcolatrice, tool_meteo, tool_wikipedia, tool_converti]

def esegui_tools(nome, parametri):
    if nome == "calcola":         return calcola(parametri["espressione"])
    if nome == "get_meteo":       return get_meteo(parametri["citta"])
    if nome == "cerca_wikipedia": return cerca_wikipedia(parametri["query"], parametri.get("lingua", "it"))
    if nome == "converti_unita":  return converti_unita(parametri["valore"], parametri["da"], parametri["a"])
    return f"Tool '{nome}' non trovato"

def chat_multi_tool(messaggio):
    history = [{"role": "user", "content": messaggio}]

    while True:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            tools=TUTTI_I_TOOL,
            messages=history
        )

        if response.stop_reason == "end_turn":
            return next(b.text for b in response.content if b.type == "text")

        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({block.input})")
                    risultato = esegui_tools(block.name, block.input)
                    tool_results.append({"type":"tool_result","tool_use_id":block.id,"content":str(risultato)})
            history.append({"role": "user", "content": tool_results})




# Prova queste domande multi-step e osserva quali tool vengono chiamati:

domande_multistep = [
    "Quanti gradi Fahrenheit fa a Sassari adesso?",  # meteo + converti
    "Qual è la distanza in miglia da Sassari a Cagliari? (sono circa 200km)",  # converti
    # Aggiungi una domanda tua che richiede 2+ tool
    "Quanto pesa in media una mucca nella capitale degli Stati Uniti considerando l'unità di misura che utilizzano li?"  # ← AGGIUNGI QUI
]

for d in domande_multistep:
    if d:
        print(f"\n❓ {d}")
        # usa chat_multi_tool con tutti i tool incluso converti
        # ...
        r = chat_multi_tool(d)
        print(f"🤖 {r}")


❓ Quanti gradi Fahrenheit fa a Sassari adesso?
  🔧 get_meteo({'citta': 'sassari'})
  🔧 converti_unita({'valore': 27.8, 'da': 'celsius', 'a': 'fahrenheit'})
🤖 A Sassari in questo momento fa **82.04 gradi Fahrenheit** (27.8°C). 

La situazione meteo è: umidità al 41% e vento a 11.4 km/h.

❓ Qual è la distanza in miglia da Sassari a Cagliari? (sono circa 200km)
  🔧 converti_unita({'valore': 200, 'da': 'km', 'a': 'miglia'})
🤖 La distanza da Sassari a Cagliari è di circa **200 km**, che corrisponde a circa **124,27 miglia** (o approssimativamente **124 miglia**).

❓ Quanto pesa in media una mucca nella capitale degli Stati Uniti considerando l'unità di misura che utilizzano li?
  🔧 cerca_wikipedia({'query': 'mucca peso medio bovino', 'lingua': 'it'})
  🔧 cerca_wikipedia({'query': 'bovino cattle', 'lingua': 'en'})
  🔧 cerca_wikipedia({'query': 'cattle weight', 'lingua': 'en'})
  🔧 converti_unita({'valore': 550, 'da': 'kg', 'a': 'libbre'})
🤖 **Risposta:**

Una mucca adulta pesa in media **ci

In [14]:
# SOLUZIONE ESERCIZIO 2
# ESERCIZIO 2 — Multi-step con tool concatenati

TUTTI_I_TOOL = [tool_calcolatrice, tool_meteo, tool_wikipedia, tool_converti]

def esegui_tutti(nome, params):
    if nome == "calcola":         return calcola(params["espressione"])
    if nome == "get_meteo":       return get_meteo(params["citta"])
    if nome == "cerca_wikipedia": return cerca_wikipedia(params["query"], params.get("lingua", "it"))
    if nome == "converti_unita":  return converti_unita(params["valore"], params["da"], params["a"])
    return f"Tool '{nome}' non trovato"

domande_multistep = [
    "Quanti gradi Fahrenheit fa a Sassari adesso?",
    "Qual è la distanza in miglia da Sassari a Cagliari? (sono circa 200km)",
    "Chi ha fondato Anthropic e quanti anni fa è stato (siamo nel 2026)?",  # Wikipedia + calcolatrice
]

for d in domande_multistep:
    if d:
        print(f"\n{'='*55}")
        print(f"❓ {d}")
        risposta = chat_con_tool_router(d, TUTTI_I_TOOL, esegui_tutti)
        print(f"🤖 {risposta}")



❓ Quanti gradi Fahrenheit fa a Sassari adesso?
  🔧 get_meteo({'citta': 'sassari'})
  ✅ Meteo a Sassari ora: 27.8°C, umidità 41%, vento 11.4 km/h. Prossimi 3 giorni: oggi max 28.4°C min 15.6°C, domani max 27.9°C, dopodomani max 25.9°C.
  🔧 converti_unita({'valore': 27.8, 'da': 'celsius', 'a': 'fahrenheit'})
  ✅ 27.8 celsius = 82.0400 fahrenheit
🤖 A Sassari adesso fa **82.04 gradi Fahrenheit** (27.8°C).

❓ Qual è la distanza in miglia da Sassari a Cagliari? (sono circa 200km)
  🔧 converti_unita({'valore': 200, 'da': 'km', 'a': 'miglia'})
  ✅ 200 km = 124.2742 miglia
🤖 La distanza tra Sassari e Cagliari è di circa **124,27 miglia** (considerando i 200 km che hai indicato).

Per essere più precisi, la distanza stradale effettiva tra Sassari e Cagliari è di circa 280 km, che corrisponde a circa **173,98 miglia**.

❓ Chi ha fondato Anthropic e quanti anni fa è stato (siamo nel 2026)?
  🔧 cerca_wikipedia({'query': 'Anthropic', 'lingua': 'it'})
  ✅ Wikipedia (Anthropic): Anthropic è una benef

### Esercizio 3 — Tool con validazione e gestione errori ★★☆
Aggiungi al tool meteo un controllo: se la città non è nella lista, invece di restituire un errore, usa Wikipedia per cercare le coordinate della città. Implementa la fallback logic.

In [15]:
# ESERCIZIO 3
def get_meteo_v2(citta: str) -> str:
    """Meteo con fallback su Wikipedia per città non in lista."""
    citta = citta.lower().strip()

    if citta in CITTA_COORD:
        return get_meteo(citta)  # versione originale

    # TODO: se la città non è in lista:
    # 1. Cerca le coordinate su Wikipedia o OpenStreetMap
    # 2. Usa quelle coordinate per chiamare open-meteo
    # Suggerimento: usa l'API Nominatim di OpenStreetMap (gratuita)
    # https://nominatim.openstreetmap.org/search?q=CITTA&format=json



    try:

        url_nominatim = "https://nominatim.openstreetmap.org/search"
        parametri_geo = {"q": citta, "format": "json", "limit": 1}
        headers = {"User-Agent": "Stefano/1.0"}

        risposta_geo = requests.get(url_nominatim, params=parametri_geo, headers=headers, timeout=5)
        dati_geo = risposta_geo.json()


        if not dati_geo:
            return f"Città '{citta}' non trovata."


        lat = dati_geo[0]["lat"]
        lon = dati_geo[0]["lon"]

        url_meteo = "https://api.open-meteo.com/v1/forecast"
        parametri_meteo = {
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m,windspeed_10m,relative_humidity_2m",
            "timezone": "Europe/Rome"
        }

        risposta_meteo = requests.get(url_meteo, params=parametri_meteo, timeout=5)
        dati_meteo = risposta_meteo.json()["current"]

        temp = dati_meteo["temperature_2m"]
        umidita = dati_meteo["relative_humidity_2m"]
        vento = dati_meteo["windspeed_10m"]

        return f"Meteo attuale per {citta.title()}: {temp}°C | Umidità: {umidita}% | Vento: {vento} km/h"

    except Exception:
        return f"Impossibile recuperare il meteo per '{citta}'"


print(get_meteo_v2("sassari"))      # in lista
print(get_meteo_v2("venezia"))      # non in lista
print(get_meteo_v2("new york"))     # città estera
print(get_meteo_v2("xyzabc"))       # città inesistente


Meteo a Sassari ora: 27.8°C, umidità 41%, vento 11.4 km/h. Prossimi 3 giorni: oggi max 28.4°C min 15.6°C, domani max 27.9°C, dopodomani max 25.9°C.
Meteo attuale per Venezia: 25.2°C | Umidità: 71% | Vento: 9.4 km/h
Meteo attuale per New York: 21.9°C | Umidità: 45% | Vento: 14.8 km/h
Città 'xyzabc' non trovata.


In [16]:
# SOLUZIONE ESERCIZIO 3
def get_meteo_v2(citta: str) -> str:
    """Meteo con fallback su Nominatim per città non in lista."""
    citta = citta.lower().strip()

    # Città già in lista — usa la versione originale
    if citta in CITTA_COORD:
        return get_meteo(citta)

    # Fallback: cerca le coordinate su Nominatim (OpenStreetMap)
    try:
        nominatim_url = "https://nominatim.openstreetmap.org/search"
        r = requests.get(
            nominatim_url,
            params={"q": citta, "format": "json", "limit": 1},
            headers={"User-Agent": "WiDataChatbot/1.0 (corso ITS Novitas)"},
            timeout=5
        )

        if r.status_code != 200 or not r.json():
            return f"Città '{citta}' non trovata"

        risultato = r.json()[0]
        lat = float(risultato["lat"])
        lon = float(risultato["lon"])
        nome_trovato = risultato.get("display_name", citta).split(",")[0]

        # Usa le coordinate per chiamare open-meteo
        url = (
            f"https://api.open-meteo.com/v1/forecast"
            f"?latitude={lat}&longitude={lon}"
            f"&current=temperature_2m,windspeed_10m,relative_humidity_2m"
            f"&timezone=Europe/Rome"
        )
        r_meteo = requests.get(url, timeout=5)

        if r_meteo.status_code != 200:
            return f"Errore meteo per '{citta}'"

        curr = r_meteo.json()["current"]
        return (
            f"Meteo a {nome_trovato.title()}: {curr['temperature_2m']}°C, "
            f"umidità {curr['relative_humidity_2m']}%, "
            f"vento {curr['windspeed_10m']} km/h."
        )

    except requests.Timeout:
        return f"Errore: timeout cercando '{citta}'"
    except Exception as e:
        return f"Errore: {str(e)}"

# Test
print(get_meteo_v2("sassari"))      # in lista — usa originale
print(get_meteo_v2("venezia"))      # non in lista — usa Nominatim
print(get_meteo_v2("new york"))     # città estera
print(get_meteo_v2("xyzabc"))       # città inesistente

Meteo a Sassari ora: 27.8°C, umidità 41%, vento 11.4 km/h. Prossimi 3 giorni: oggi max 28.4°C min 15.6°C, domani max 27.9°C, dopodomani max 25.9°C.
Meteo a Venezia: 25.2°C, umidità 71%, vento 9.4 km/h.
Meteo a New York: 21.9°C, umidità 45%, vento 14.8 km/h.
Città 'xyzabc' non trovata


### Esercizio 4 — Chatbot completo con tool + RAG + MCP ★★★ (Deliverable!)

Integra tutto quello che hai costruito nelle lezioni 3, 4 e 5:
- Conversation history (sliding window)
- RAG sul documento WiData
- Tool calcolatrice, meteo, Wikipedia
- MCP server filesystem
- Streaming
- System prompt WiData

In [17]:
!pip install chromadb sentence-transformers -q

In [18]:
# Simulazione del pattern MCP — come un server MCP espone i tool

class MockMCPServer:
    """Simula un server MCP per il filesystem locale."""

    def list_tools(self):
        """Restituisce la lista dei tool disponibili (come farebbe un server MCP reale)."""
        return [
            {
                "name": "read_file",
                "description": "Leggi il contenuto di un file di testo",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "Percorso del file"}
                    },
                    "required": ["path"]
                }
            },
            {
                "name": "list_files",
                "description": "Elenca i file in una directory",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "directory": {"type": "string", "description": "Percorso della directory"}
                    },
                    "required": ["directory"]
                }
            }
        ]

    def call_tool(self, nome, parametri):
        """Esegue un tool e restituisce il risultato."""
        import os
        if nome == "read_file":
            path = parametri["path"]
            if os.path.exists(path):
                with open(path, "r", encoding="utf-8") as f:
                    content = f.read()
                return content[:1000]  # max 1000 caratteri
            return f"File non trovato: {path}"
        elif nome == "list_files":
            directory = parametri["directory"]
            if os.path.exists(directory):
                files = os.listdir(directory)
                return f"File in '{directory}': {', '.join(files[:20])}"
            return f"Directory non trovata: {directory}"
        return f"Tool '{nome}' non disponibile"


In [24]:
# ESERCIZIO 4 — Chatbot completo (DELIVERABLE)
# Integra: RAG (L4) + Tool use (L5) + History + Streaming

import chromadb

chroma_client = chromadb.Client()
mcp_server = MockMCPServer()

mcp_tools = mcp_server.list_tools()
TUTTI_I_TOOL = [tool_calcolatrice, tool_meteo, tool_wikipedia, tool_converti]
tutti_i_tool_con_mcp = TUTTI_I_TOOL + mcp_tools

MAX_MESSAGGI = 10




SYSTEM_WIDATA_COMPLETO = """
                            Sei l'assistente virtuale di WiData Srl, startup IoT e smart cities di Sassari.
                            Hai accesso a:
                            - Documenti WiData (via RAG) per informazioni sui prodotti
                            - Tool meteo per informazioni meteo in tempo reale
                            - Tool Wikipedia per informazioni generali
                            - Tool calcolatrice per calcoli precisi
                            - File system per leggere note e documenti locali

                            Usa i tool quando appropriato. Per i prodotti WiData, usa SEMPRE il contesto RAG.
                            Se non trovi una risposta, dillo chiaramente.
                        """




def chunka_testo(testo, chunk_size=400, overlap=50):
    """Divide il testo in chunk con overlap."""
    chunks = []
    start = 0
    while start < len(testo):
        end = start + chunk_size
        chunk = testo[start:end]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks



def setup_rag(testo):
    """Indicizza il documento e restituisce la collection."""
    nome_collection = "info_widata"
    chunks = chunka_testo(testo)
    collection_widata = chroma_client.get_or_create_collection(name=nome_collection, metadata={"hnsw:space": "cosine"})
    collection_widata.add(documents=chunks, ids=[f"chunk_{i}" for i in range(len(chunks))])
    print(f"Sono stati indicizzati {collection_widata.count()} chunk.")
    return collection_widata




def cerca(domanda, collection, n=2):
    risultati = collection.query(
        query_texts=[domanda],
        n_results=n
    )
    return risultati["documents"][0]




def esegui_tool_completo(nome, parametri):
    """Router che gestisce sia tool custom che tool MCP."""

    if nome == "calcola":           return calcola(parametri["espressione"])
    if nome == "get_meteo":         return get_meteo_v2(parametri["citta"])
    if nome == "cerca_wikipedia":   return cerca_wikipedia(parametri["query"], parametri.get("lingua", "it"))
    if nome == "converti_unita":  return converti_unita(parametri["valore"], parametri["da"], parametri["a"])

    if nome in ["read_file", "list_files"]:
        return mcp_server.call_tool(nome, parametri)
    return f"Tool '{nome}' non trovato"












def chatbot_completo(domanda, history, collection):
    """Chatbot con RAG + tool + storia + streaming."""
    # TODO:
    # 1. Recupera chunk RAG rilevanti
    # 2. Costruisci il messaggio con contesto RAG
    # 3. Aggiungi alla history
    # 4. Chiama l'API con tools= e history
    # 5. Gestisci il tool loop
    # 6. Streaming sulla risposta finale

    chunks_rilevanti = []
    if collection is not None:
      chunks_rilevanti = cerca(domanda, collection)

    if chunks_rilevanti:
      contesto = "\n---\n".join(chunks_rilevanti)

      prompt = f"""Documenti di riferimento:
                          {contesto}
                              ---
                  Domanda dell'utente: {domanda}"""
    else:
      prompt = domanda

    history.append({"role": "user", "content": prompt})

    print("\n🤖 Assistente: ", end="", flush=True)


    while True:

      response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=800,
            system=SYSTEM_WIDATA_COMPLETO,
            tools=tutti_i_tool_con_mcp,
            messages=history[-MAX_MESSAGGI:]
        )



      if response.stop_reason == "tool_use":
          history.append({"role": "assistant", "content": response.content})
          tool_results = []
          for block in response.content:
              if block.type == "tool_use":
                  print(f"  🔧 {block.name}({block.input})")
                  risultato = esegui_tool_completo(block.name, block.input)
                  print(f"  ✅ {str(risultato)[:100]}")
                  tool_results.append({
                      "type": "tool_result",
                      "tool_use_id": block.id,
                      "content": str(risultato)
                  })
          history.append({"role": "user", "content": tool_results})
          continue


      if response.stop_reason == "end_turn":
          testo_completo = ""


          print("\n🤖 ", end="", flush=True)
          with client.messages.stream(
              model="claude-haiku-4-5-20251001",
              max_tokens=800,
              system=SYSTEM_WIDATA_COMPLETO,
              messages=history[-MAX_MESSAGGI:]
          ) as stream:
              for text in stream.text_stream:
                  testo_completo += text
                  print(text, end="", flush=True)
          print("\n")
          history.append({"role": "assistant", "content": testo_completo})


      return testo_completo







def main():

    documento_widata = """
                      WiData Srl — Manuale Prodotti IoT

                      SENSORE XS200 - MONITORAGGIO AMBIENTALE
                      Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
                      Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica e qualità dell'aria (CO2, PM2.5).
                      Classificazione IP67: impermeabile e resistente alla polvere. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
                      Connettività: LoRaWAN, NB-IoT, WiFi 802.11n. Dimensioni: 85x45x30mm. Peso: 120g.
                      Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

                      GATEWAY GW500 - CONCENTRATORE DATI
                      Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
                      Copertura fino a 15km in aree rurali, 3km in aree urbane. Elaborazione edge computing integrata.
                      Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
                      Alimentazione: 220V AC o pannello solare (opzionale). Temperatura operativa: -40°C a +70°C.
                      Certificazioni: CE, IP65. Installazione: palo, tetto o rack.

                      PIATTAFORMA XPLORE - ANALYTICS
                      Xplore è la piattaforma cloud di WiData per la visualizzazione e analisi dei dati IoT.
                      Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
                      Alerting automatico via email, SMS o webhook quando i valori superano soglie configurabili.
                      API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
                      Machine learning per previsione anomalie e manutenzione predittiva.
                      Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
                      SLA: 99.9% uptime garantito nei piani Pro ed Enterprise.

                      SUPPORTO E ASSISTENZA
                      Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
                      Email: support@widata.cloud | Telefono: +39 079 123456.
                      Per informazioni commerciali: sales@widata.cloud.
                      Sede: Via Roma 42, Sassari (SS) 07100, Italia.
                      Spedizioni in tutta Italia in 3-5 giorni lavorativi.
                  """






    collection = setup_rag(documento_widata)
    history = []
    print("🤖 Chatbot WiData RAG avviato. Digita 'esci' per uscire.\n")

    while True:
        utente = input("Tu: ")
        if utente.lower() == "esci":
            print("👋 Arrivederci!")
            break
        chatbot_completo(utente, history, collection)








main()  # Decommentare per eseguire

Sono stati indicizzati 7 chunk.
🤖 Chatbot WiData RAG avviato. Digita 'esci' per uscire.

Tu: ciao mi chiamo Stefano

🤖 Assistente: 
🤖 Ciao Stefano! 👋 

Benvenuto! Sono l'assistente virtuale di **WiData Srl**, la startup IoT e smart cities di Sassari.

Sono qui per aiutarti con:
- 📊 **Informazioni sui nostri prodotti** (sensori IoT, piattaforme di monitoraggio, machine learning per manutenzione predittiva)
- 🌤️ **Dati meteo** in tempo reale
- 🔧 **Supporto tecnico** e informazioni generali
- 📱 **Contatti commerciali** e amministrativi

**Come posso aiutarti oggi?** Hai domande sui nostri servizi, vuoi conoscere i nostri piani tariffari, o c'è altro in cui posso assisterti?

Tu: che tempo fa a sassari?

🤖 Assistente:   🔧 get_meteo({'citta': 'sassari'})
  ✅ Meteo a Sassari ora: 24.9°C, umidità 47%, vento 9.1 km/h. Prossimi 3 giorni: oggi max 28.3°C min 15.

🤖 A Sassari adesso:
- 🌡️ **Temperatura**: 24.9°C
- 💧 **Umidità**: 47%
- 💨 **Vento**: 9.1 km/h

**Previsione prossimi 3 giorni:**
- **O

In [28]:
# SOLUZIONE ESERCIZIO 4
# ESERCIZIO 4 — Chatbot completo (DELIVERABLE)
# Integra: RAG (L4) + Tool use (L5) + History + Streaming



documento_widata = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica e qualità dell'aria (CO2, PM2.5).
Classificazione IP67: impermeabile e resistente alla polvere. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n. Dimensioni: 85x45x30mm. Peso: 120g.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare (opzionale). Temperatura operativa: -40°C a +70°C.
Certificazioni: CE, IP65. Installazione: palo, tetto o rack.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook quando i valori superano soglie configurabili.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Machine learning per previsione anomalie e manutenzione predittiva.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptime garantito nei piani Pro ed Enterprise.

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Per informazioni commerciali: sales@widata.cloud.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
"""

SYSTEM_WIDATA_COMPLETO = """
Sei l'assistente virtuale di WiData Srl, startup IoT e smart cities di Sassari.
Hai accesso a:
- Documenti WiData (via RAG) per informazioni sui prodotti
- Tool meteo per informazioni meteo in tempo reale
- Tool Wikipedia per informazioni generali
- Tool calcolatrice per calcoli precisi

Usa i tool quando appropriato. Per i prodotti WiData, usa SEMPRE il contesto RAG.
Se non trovi una risposta, dillo chiaramente.
"""

TUTTI_I_TOOL = [tool_calcolatrice, tool_meteo, tool_wikipedia, tool_converti]

def esegui_tool_completo(nome, params):
    if nome == "calcola":         return calcola(params["espressione"])
    if nome == "get_meteo":       return get_meteo_v2(params["citta"])
    if nome == "cerca_wikipedia": return cerca_wikipedia(params["query"], params.get("lingua", "it"))
    if nome == "converti_unita":  return converti_unita(params["valore"], params["da"], params["a"])
    return f"Tool '{nome}' non trovato"

def chatbot_completo(messaggio, history, collection):
    """Chatbot con RAG + tool + storia + streaming."""

    # 1. Recupera chunk RAG rilevanti
    chunks = []
    if collection is not None:
        risultati = collection.query(query_texts=[messaggio], n_results=3)
        chunks = risultati["documents"][0]

    # 2. Costruisci il messaggio con contesto RAG
    if chunks:
        contesto = "\n\n---\n\n".join(chunks)
        messaggio_con_rag = f"""Documenti di riferimento:

{contesto}

---

Domanda: {messaggio}"""
    else:
        messaggio_con_rag = messaggio

    # 3. Aggiungi alla history
    history.append({"role": "user", "content": messaggio_con_rag})

    # 4 + 5. Tool loop
    iterazioni = 0
    while True:
        iterazioni += 1
        if iterazioni > 10:
            print("⚠️ Loop non terminato")
            break

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=800,
            system=SYSTEM_WIDATA_COMPLETO,
            tools=TUTTI_I_TOOL,
            messages=history
        )

        # Tool richiesto — esegui e rimanda
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({block.input})")
                    risultato = esegui_tool_completo(block.name, block.input)
                    print(f"  ✅ {str(risultato)[:100]}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(risultato)
                    })
            history.append({"role": "user", "content": tool_results})
            continue

        # 6. Risposta finale — streaming
        if response.stop_reason == "end_turn":
            history.append({"role": "assistant", "content": response.content})
            testo_completo = ""

            # Richiama con streaming per la risposta finale
            print("\n🤖 ", end="", flush=True)
            with client.messages.stream(
                model="claude-haiku-4-5-20251001",
                max_tokens=800,
                system=SYSTEM_WIDATA_COMPLETO,
                messages=history[:-1]  # escludi l'ultima risposta già generata
            ) as stream:
                for text in stream.text_stream:
                    testo_completo += text
                    print(text, end="", flush=True)
            print("\n")

            # Aggiorna l'ultimo messaggio in history con il testo completo
            history[-1] = {"role": "assistant", "content": testo_completo}
            return testo_completo

def main():
    # Setup ChromaDB
    import chromadb
    chroma_client = chromadb.Client()
    collection = chroma_client.get_or_create_collection("widata_main")

    # Indicizza il documento WiData se vuoto
    if collection.count() == 0:
        chunks = chunka_testo(documento_widata)
        collection.add(documents=chunks, ids=[str(i) for i in range(len(chunks))])
        print(f"✅ {collection.count()} chunk indicizzati\n")

    history = []
    print("🤖 Chatbot WiData avviato. Digita 'esci' per uscire.\n")

    while True:
        try:
            utente = input("Tu: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Arrivederci!")
            break

        if not utente:
            continue
        if utente.lower() == "esci":
            print("👋 Arrivederci!")
            break

        chatbot_completo(utente, history, collection)

main()

✅ 6 chunk indicizzati

🤖 Chatbot WiData avviato. Digita 'esci' per uscire.

Tu: cos'è Xplore?

🤖 # Xplore - Piattaforma Analytics di WiData

**Xplore** è la piattaforma cloud di WiData dedicata alla visualizzazione e analisi dei dati IoT.

## Caratteristiche principali:

📊 **Dashboard personalizzabili** con grafici in tempo reale

📈 **Storico dati** fino a 5 anni

🚨 **Alerting automatico** via email, SMS o webhook quando i valori superano soglie configurabili

🔗 **API REST** per integrazione con sistemi terzi (ERP, SCADA, BIM)

🤖 **Machine Learning** per previsione anomalie e manutenzione predittiva

## Piani disponibili:

- **Free**: 5 sensori (gratuito)
- **Pro**: 100 sensori, €49/mese con SLA 99.9% uptime garantito
- **Enterprise**: sensori illimitati, prezzo su richiesta con SLA 99.9% uptime garantito

Xplore è la soluzione ideale se hai bisogno di monitorare, analizzare e gestire i tuoi dati IoT da una piattaforma centralizzata e potente.

Hai domande specifiche su come utilizzarl

---
## 📤 Consegna

1. Completa tutti gli esercizi
2. Scarica: `File → Scarica → .ipynb`
3. Rinomina: `Lezione5_TUONOME.ipynb`
4. Carica su GitHub in `lezione5/`

```bash
git add lezione5/
git commit -m "Lezione 5 completata"
git push
```

---
### 📖 Per la prossima lezione (Martedì 09/06)
Leggi **Huyen Cap. 3 + 4** (Evaluation) e **Cap. 10** (Architecture)

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*